# 08 · Capstone: ShipFast and the research agent in LangGraph

Notebooks 01 to 07 built the pieces. This notebook puts them together into the two agents every
framework in this project implements against the same contract (`contracts/`), and shows how they are
evaluated. The agents live in `lg_agents/`, and `run_eval.py` runs the full evaluation with the same code.

**You will see**

- the ShipFast agent's design in LangGraph, and why its Resolution is built in code
- two hard ShipFast scenarios run live through the **shared runner**: an approval, and a two-turn case
- one research question through the shared research runner
- this framework's full results, and all three frameworks side by side

In [1]:
import lg_agents.shipfast as sf
from lg_agents.research_flow import answer
from lg_agents.shipfast import LangGraphShipFast

from agent_contracts.harness import verify_model
from agent_contracts.harness.runners import run_question, run_scenario
from agent_contracts.research.scoring import load_questions
from agent_contracts.shipfast.scoring import load_scenarios

verify_model()
scenarios = {s["id"]: s for s in load_scenarios()}
questions = {q["id"]: q for q in load_questions()}

## The ShipFast agent

In LangGraph the agent is a `StateGraph` (triage node, conditional edge, a fixed decline node, and the `create_agent`
specialist added as a subgraph with the approval middleware), with a checkpointer per conversation thread
that also holds a paused approval.

Every build fills in the scored `Resolution` the same way, with the shared
`agent_contracts.shipfast.resolution.derive_resolution` (DECISIONS.md 16): `intent` and `order_id` from
triage; `action`, `refund_amount` and `ticket_id` from what actually happened in the ledger and the
tickets; and only when a turn left no record does the model choose between `answered`, `needs_info` and
`declined`. The component notebooks showed why: asked to fill in a Resolution itself, the 8B model
claimed refunds that never happened and invented ticket numbers.

In [2]:
print(sf.__doc__)

The ShipFast support agent in LangGraph (ShipFast SPEC.md), the capstone of notebooks 01 to 07.

    START -> triage --(intent == other)--> decline -> END
                    \-----(otherwise)----> specialist -> END

- triage: structured `Triage` from the conversation so far (shared `prompts.TRIAGE`).
- decline: a fixed reply, no model call (policy P7).
- specialist: the `create_agent` loop with the shared tools and instructions, added to the graph as a
  subgraph, with the $100 approval middleware (`interrupt()` before a large refund).
- A checkpointer keeps each conversation (thread) and holds a paused approval until it is resumed.

The Resolution is built by the shared `derive_resolution` from the ledger and tickets (DECISIONS.md 16);
the model only writes the reply and, when a turn left no record, picks the `Outcome`.



In [3]:
for e in sf.build_shipfast().get_graph().edges:
    print(f"  {e.source:10} -> {e.target:10} {'(conditional)' if e.conditional else ''}")

  __start__  -> triage     
  triage     -> decline    (conditional)
  triage     -> specialist (conditional)
  decline    -> __end__    
  specialist -> __end__    


## Run through the shared runner

`run_scenario` is the same function every framework's `run_eval.py` uses. It resets the shared tools,
feeds each turn, resumes a paused approval with the scenario's decision, and checks the result against
`scenarios.json`. The framework only supplies the adapter: start a conversation, send a turn, resume.

In [4]:
def show(item):
    print(f"{item['id']}: {'PASS' if item['passed'] else 'FAIL'}  ({item['latency_s']:.0f}s, "
          f"{item['llm_calls']} model calls, tools {item['tool_calls']})")
    for i, t in enumerate(item["turn_resolutions"], 1):
        print(f"  turn {i}: {t['intent']} {t['order_id']} -> {t['action']} {t['refund_amount'] or ''}")
        print(f"          {t['reply'].strip()[:160]}")
    for p in item["problems"]:
        print("  problem:", p)


show(await run_scenario(LangGraphShipFast(), scenarios["S3a_late_refund_approved"]))

S3a_late_refund_approved: PASS  (52s, 6 model calls, tools ['lookup_order', 'track_shipment', 'search_policy', 'issue_refund'])
  turn 1: refund_request A1003 -> refunded 250.0
          Your refund for order A1003 has been issued. You will receive a full refund of $250.00. Thank you for your patience and understanding.


S3a is the late espresso machine, a $250 refund the supervisor approves. A passing run pauses at the
approval gate, is resumed with `approve`, and issues exactly $250.

In [5]:
show(await run_scenario(LangGraphShipFast(), scenarios["S4_damaged_needs_photo_then_refund"]))

S4_damaged_needs_photo_then_refund: PASS  (87s, 9 model calls, tools ['lookup_order', 'track_shipment', 'search_policy', 'issue_refund'])
  turn 1: complaint A1004 -> needs_info 
          The vase set from your order A1004 arrived broken. According to our policy, you must provide a photo of the damage within 7 days of the delivery date. Since the
  turn 2: complaint A1004 -> refunded 60.0
          Your full refund of $60.00 for the broken vase set in order A1004 has been processed. You should see the refund reflected in your account within a few business 


S4 is the broken vase: the first turn must ask for a photo and refund nothing (policy P3); the second,
which never names the order, must refund the $60 total. It needs memory and policy in the right order.

In the saved run both passed. S3a paused at the approval middleware, was resumed with `approve`, and issued
exactly $250 (the runner records each turn's final state, so the pause itself is not listed as a turn).
S4 asked for the photo and refunded nothing on the first turn, then refunded $60 once on the second,
using the conversation the checkpointer kept.

## One research question

The research agent is the workflow from notebook 05, answered through the shared `run_question`. Q07 has
a stale article (9 provinces) beside the current one (14 provinces); mentioning the stale figure at all
fails the answer.

In [6]:
item = await run_question(answer, questions["Q07_deployment"])
print("passed:", item["passed"], "| tools:", item["tool_calls"], "| model calls:", item["llm_calls"], f"| {item['latency_s']:.0f}s")
print("answer:", item["report"]["answer"])
print("problems:", item["problems"] or "none")

passed: True | tools: ['web_search', 'web_search', 'web_search'] | model calls: 7 | 140s
answer: Veltara Aquatech has deployed 2,800 TidePure units across 14 provinces by the end of 2025, as reported in their 2025 annual report [F04].
problems: none


## The full evaluation

`run_eval.py` runs every scenario and every question once and writes `results/shipfast/langgraph.json` and
`results/research/langgraph.json` (from the tutorial folder: `uv run python run_eval.py`). It takes about half an
hour on the reference Mac, so this notebook reads the saved results rather than re-running them.

In [7]:
import json
from pathlib import Path

RESULTS = Path("..").resolve().parent / "results"


def load(agent, framework):
    path = RESULTS / agent / f"{framework}.json"
    return json.loads(path.read_text()) if path.exists() else None


for agent, key in (("shipfast", "scenarios"), ("research", "questions")):
    data = load(agent, "langgraph")
    print(f"== {agent}: {data['summary']['passed']} of {data['summary']['total']} passed (run at {data['run_at']})")
    for it in data[key]:
        if not it["passed"]:
            print(f"  {it['id']}: {it['problems']}")

== shipfast: 7 of 8 passed (run at 2026-11-03T08:02:55+00:00)
  S5_outside_refund_window: ["action: expected 'declined', got 'refunded'", 'refund_amount: expected None, got 35.0']
== research: 10 of 10 passed (run at 2026-11-03T08:27:30+00:00)


## All three frameworks

The same contract, data, model, settings, tools, instructions, scoring and runner, so the differences
below come from the frameworks (and from how a small model reacts to each one's way of packaging the
same request).

In [8]:
rows = []
for agent in ("shipfast", "research"):
    for framework in ("langgraph", "crewai", "strands"):
        data = load(agent, framework)
        if data:
            s = data["summary"]
            rows.append((agent, framework, f"{s['passed']}/{s['total']}", s["total_tokens"], round(s["total_latency_s"])))
print(f"{'agent':9} {'framework':10} {'passed':7} {'tokens':>7} {'seconds':>8}")
for r in rows:
    print(f"{r[0]:9} {r[1]:10} {r[2]:7} {r[3]:7} {r[4]:8}")

agent     framework  passed   tokens  seconds
shipfast  langgraph  7/8       29962      387
shipfast  crewai     7/8       34867      401
shipfast  strands    4/8       26981      331
research  langgraph  10/10     44361     1464
research  crewai     9/10      60243     1194
research  strands    9/10      48803     1147


LangGraph passed 7 of 8 ShipFast scenarios and all 10 research questions. Its one ShipFast failure, S5,
refunded the stand delivered 46 days ago: the 30-day rule is in the policy the model reads, but nothing in
code enforces it, and the model missed it, as it did in notebook 03.

Across the frameworks, LangGraph and CrewAI tie on ShipFast with different single failures; Strands is
lower, mostly because its agent twice announced the $250 refund and ended its turn without attempting it,
so the approval gate never had anything to pause. On tokens, Strands used the fewest and CrewAI the most
for both agents; LangGraph took the longest on research. These are single runs on one laptop: a first
measurement, not a ranking.

## How the others do this

- **CrewAI** builds the same agent as a **Flow**, one run per customer turn with the history in the flow's
  state; the approval pause sits between flow steps and resumes from saved state.
- **Strands** uses **one agent per conversation**, with triage as a separate structured call and the
  approval as a hook that interrupts before the refund.

## Takeaways

- The contract keeps the comparison fair: one copy of the data, tools, instructions, scoring and runner.
- Code records the facts (refunds, tickets, pauses); the model writes the reply and makes one small call.
- Every framework could build both agents, but each needed its own shape for the approval pause and for
  memory, and each made different mistakes with the same model. Those differences are the comparison.